# 01 — Exploración inicial (PySpark)

Notebook conectado al cluster Spark del proyecto.
Datos: bronze `trips` adaptado al schema del PDF.

**Pre-requisito:** haber corrido al menos la tarea `ingest` del DAG, o tener
el parquet en `/home/jovyan/data/bronze/trips/`.

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("mobility-exploration")
    .master(os.environ.get("SPARK_MASTER", "spark://mobility-spark-master:7077"))
    .config("spark.executor.memory", "2g")
    .getOrCreate()
)
spark

In [ ]:
BRONZE = "/home/jovyan/data/bronze/trips"
trips = spark.read.parquet(BRONZE)
trips.printSchema()
print("Filas:", trips.count())
trips.show(5, truncate=False)

## Inspección de calidad

In [ ]:
key_cols = ["pickup_datetime", "dropoff_datetime", "trip_distance_km", "fare_amount", "passenger_count"]
nulls = trips.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in key_cols
])
nulls.show()

In [ ]:
trips.describe(["trip_distance_km", "fare_amount", "passenger_count"]).show()

## Distribución por hora del día

In [ ]:
by_hour = (
    trips.withColumn("hour", F.hour("pickup_datetime"))
         .groupBy("hour").count()
         .orderBy("hour")
         .toPandas()
)
ax = by_hour.plot.bar(x="hour", y="count", legend=False, figsize=(10, 4))
ax.set_xlabel("Hora"); ax.set_ylabel("Viajes"); ax.set_title("Demanda por hora (bronze)")

## Outliers candidatos
Estos son los registros que `clean.py` descarta.

In [ ]:
outliers = trips.filter(
    (F.col("trip_distance_km") <= 0)
    | (F.col("fare_amount") <= 0)
    | (F.col("passenger_count") <= 0)
)
print("Outliers:", outliers.count())
outliers.select("trip_distance_km", "fare_amount", "passenger_count").show(10)

In [ ]:
spark.stop()